# 第 12 课 · 知道明天下雨就重估单量（浏览器版）

配套课文：[docs/m0-foundations/12-conditioning-and-control.md](../docs/m0-foundations/12-conditioning-and-control.md)。

**怎么用**：本地装了 Jupyter 就直接打开这一页；没有环境就上 https://jupyter.org/try （JupyterLite）或 Colab，把这份 notebook 整份放进去，每一格按 `Shift + Enter` 运行。只用 Python 标准库，不需要安装任何东西。

每格跑出来的数字和课文里印的逐字一致（种子钉在 7）。有两格要跑两万遍循环，会多等一两秒，次数是钉死的，别改——改了数字就对不上了。


## 一、你模拟的时候，把已经知道的事又抽了一遍

你在经营一家小店，中午外卖。你手上有一条规律：下雨天单量平均 60，晴天平均 40，本地一个月大概三成天在下雨。

按第 09 课的做法，你写了一个模拟：循环三十次，每次先抽"今天下不下雨"，再抽"今天多少单"，加起来就是月总量。跑两万遍，报出一个平均和三个分位数。

这段代码没错。但请你注意一件事：**如果下个月的排班表已经写死了、天气预报已经给了你十二天雨，你却让电脑每天重新抽一次"下不下雨"——你是在花随机数去买你已经知道的东西。**

更常见的是另一种：你连"下雨"这件事都当成随机，其实你那个月本来就有几件确定要发生的事（固定的活动、签了的服务、日历上的节假日）。它们不是随机的，可它们在代码里是随机的。

这一节课就处理这个：**知道一件事之后，另一件事该怎么重算。** 它有两个结果——期望会变，误差也会变，而且后一个变得往往很大。


先把这个月世界的参数和两台机器搭好（下面这一格只是定义，运行它不会打印任何东西）：五个数在最上面，改成你自己的场景就动它们。


In [1]:
import math
import random

RAIN_P = 0.3
RAIN_MEAN, RAIN_SD = 60.0, 8.0
FAIR_MEAN, FAIR_SD = 40.0, 5.0
DAYS = 30


def spread(vals):
    n = len(vals)
    m = sum(vals) / n
    return math.sqrt(sum((x - m) ** 2 for x in vals) / (n - 1))


def one_month_random_weather():
    """老老实实模拟一个月：每天连天气一起抽。"""
    total = 0
    for _ in range(DAYS):
        if random.random() < RAIN_P:
            total += random.gauss(RAIN_MEAN, RAIN_SD)
        else:
            total += random.gauss(FAIR_MEAN, FAIR_SD)
    return total


def one_month_known_weather(rain_days):
    """你已经知道这个月有几天雨：天气这一层不再抽，只抽每天单量。"""
    total = 0
    for _ in range(rain_days):
        total += random.gauss(RAIN_MEAN, RAIN_SD)
    for _ in range(DAYS - rain_days):
        total += random.gauss(FAIR_MEAN, FAIR_SD)
    return total


def control(rho, runs):
    """要估的量 Y，和一个你已经知道平均值的辅助量 X，两者相关 rho。
    X 平均固定为 0，所以"减掉 a 倍的 (X − 0)"是合法的：它不减掉真值，只减掉噪声。"""
    plain = []
    fixed = []
    for _ in range(runs):
        common = random.gauss(0, 1)
        x = common
        y = rho * common + math.sqrt(1 - rho ** 2) * random.gauss(0, 1)
        plain.append(y)
        fixed.append(y - rho * (x - 0))
    return spread(plain), spread(fixed)


## 二、把期望拆开算

先只算期望，不碰模拟。

你要的是"平均一天多少单"。你知道的是两件事各自的条件：雨天平均 60，晴天平均 40，雨天占三成。那么：

$$E[\text{单日单量}] = 0.3 \times 60 + 0.7 \times 40 = 46.0$$

这个式子替你回答的问题很直白：**"我不知道今天下不下雨，那我该用哪个数？"** 答案是"按可能性给两个平均掺一下"。它是第 06 课期望定义的直接套用——把每种情况下的平均值，按那种情况的机会加权。

一个月就是 $30 \times 46.0 = 1380$ 单。让电脑老老实实按"先抽天气、再抽单量"演两万遍核对：


In [2]:
per_day = RAIN_P * RAIN_MEAN + (1 - RAIN_P) * FAIR_MEAN
print(f"雨天 {RAIN_MEAN:.0f} 单、晴天 {FAIR_MEAN:.0f} 单、下雨概率 {RAIN_P}：")
print(f"拆开来算的日期望 = {RAIN_P} × {RAIN_MEAN:.0f} + {1 - RAIN_P} × {FAIR_MEAN:.0f} = {per_day:.1f}")
print(f"一个月（{DAYS} 天）= {DAYS * per_day:.0f} 单")
random.seed(7)
months = [one_month_random_weather() for _ in range(20000)]
print(f"让电脑演两万遍：平均 {sum(months) / len(months):.1f} 单（和上面同一个数）")


雨天 60 单、晴天 40 单、下雨概率 0.3：
拆开来算的日期望 = 0.3 × 60 + 0.7 × 40 = 46.0
一个月（30 天）= 1380 单
让电脑演两万遍：平均 1380.0 单（和上面同一个数）


两条路对上同一个数。

这里冒出一个新说法。**"已知下雨时的平均单量"写成 $E[Y \mid X]$，读作"在 X 之下的 Y 期望"，竖线就是"已经知道"的意思。** 它叫**条件期望**。上面那一行其实是它的组合版：先有"雨天 60、晴天 40"这两个条件期望，再按天气的机会加权回去，得到不带条件的期望。这个"先分后合"的规矩叫**全期望公式**。

你现在只需要记住：**条件期望就是"知道了一件事之后，你对另一件事的平均要换成哪个数"。** 它不是新算法，是你本来就会的加权平均。


## 三、一条信息要重算两笔账

现在把"你已经知道"这件事真的放进代码：假设这个月**确定**有 12 天雨（预报给你的、或者你的排班决定了的），你不再抽天气，只抽每天单量。跑两万遍，和原来那版对比：


In [3]:
random.seed(7)
blind = [one_month_random_weather() for _ in range(20000)]
rain_days = 12
random.seed(7)
foreseen = [one_month_known_weather(rain_days) for _ in range(20000)]
print(f"不知道天气、每天现抽：月总量 平均 {sum(blind) / len(blind):.1f}，起伏 {spread(blind):.1f} 单")
print(f"已知这个月有 {rain_days} 天雨：月总量 平均 {sum(foreseen) / len(foreseen):.1f}，起伏 {spread(foreseen):.1f} 单")
print(f"期望多了 {sum(foreseen) / len(foreseen) - sum(blind) / len(blind):.1f} 单，起伏少了 {(1 - spread(foreseen) / spread(blind)) * 100:.1f}%")
print(f"（前提是这 {rain_days} 天是你真知道的：预报、排班、合同。不能是 30 × 0.3 算出来的。）")


不知道天气、每天现抽：月总量 平均 1380.0，起伏 59.7 单
已知这个月有 12 天雨：月总量 平均 1440.1，起伏 34.8 单
期望多了 60.1 单，起伏少了 41.7%
（前提是这 12 天是你真知道的：预报、排班、合同。不能是 30 × 0.3 算出来的。）


两笔账都变了：

**期望从 1380 涨到 1440。** 因为你知道这个月雨多，12 天而不是平均 9 天。这条不用解释——信息当然会改你的判断。

**可毛边从 59.7 掉到 34.8，少了 41.7%。** 这一条才是这一节课的主角：**你多用了一条信息，什么都没多算，误差就少了四成。** 而你要做的只是把"抽天气"那一行删掉。

顺手说一句怎么读这个数：在"已知 12 天雨"的条件下，你的月总量大概是 1440，毛边 35，所以你敢报的范围是 1440 ± 70（第 07、08 课那套乘二）。而不知道天气那一版，范围是 1380 ± 120。**想把后者的毛边也压到 35 附近，你得把模拟次数加到大约三倍**（$59.7/\sqrt{3} \approx 34.5$，就是第 07 课那笔根号税）——而那条预报不花一份算力，还顺手把期望修正了 60 单。**多演只在"你已经把该知道的都知道了"之后才值钱。**


## 四、为什么能降这么多：大头在"组间"

第三节的毛边降四成不是运气。这一节给你看它从哪来，因为**知道这个来源，你就知道什么样的已知信息值钱。**

一个量的总起伏可以拆成两块：一块是**组内**（雨天和晴天各自内部还会晃），把组定下来也消不掉；另一块是**组间**（今天到底下不下雨），你一旦知道它就彻底消失。

$$\text{总起伏（方差）} = \text{组内} + \text{组间}$$

它替你回答的问题是：**"我的误差是从哪来的？我该去找哪一类信息？"** 按我们的数算一遍（`blind` 是第三节那一轮还在的名字，用来对照）：


In [4]:
within = RAIN_P * RAIN_SD ** 2 + (1 - RAIN_P) * FAIR_SD ** 2
between = RAIN_P * (RAIN_MEAN - per_day) ** 2 + (1 - RAIN_P) * (FAIR_MEAN - per_day) ** 2
print(f"单日：组内（雨天/晴天各自内部还晃）= {within:.1f}")
print(f"单日：组间（今天到底下不下雨）= {between:.1f}")
print(f"加起来 = {within + between:.1f}；组间占 {(between / (within + between)) * 100:.1f}%")
print(f"一个月（{DAYS} 天）：{(within + between) * DAYS:.0f}，开根号 = {math.sqrt((within + between) * DAYS):.1f} 单")
print(f"和第二部分实测的 {spread(blind):.1f} 单是同一个数。")


单日：组内（雨天/晴天各自内部还晃）= 36.7
单日：组间（今天到底下不下雨）= 84.0
加起来 = 120.7；组间占 69.6%
一个月（30 天）：3621，开根号 = 60.2 单
和第二部分实测的 59.7 单是同一个数。


按数算一遍：单日组内 = $0.3 \times 8^2 + 0.7 \times 5^2 = 36.7$；单日组间 = $0.3 \times (60-46)^2 + 0.7 \times (40-46)^2 = 84.0$。加起来 120.7，乘三十天开根号 = **60.2 单**——和第三节实测的 59.7 是同一个数（差在抽签毛边）。

而 **60.2 里有 69.6% 来自组间**。这就是第三节那 41.7% 的全部来历：你一旦把"这个月几天下雨"定下来，那块占七成的方差就不再参与抽签了，只剩下组内那部分（三成）还得靠模拟。

这条规律很实用，因为它给了你一个可以提前问出口的问题：**"我这块误差，是'组内'的还是'组间'的？"** 组间的大头，靠**信息**解决；组内的大头，才靠**多演几遍**解决。多数人把预算花错了地方——他们把已经能问清楚的事（促销有几天、几个大客户会下单、这季度发几条）留在代码里当随机数，然后花十倍算力去抽它。

这也正好是第 11 课那个共同因素的另一种说法：`common` 那一层（下不下雨、大环境好不好）是组间，各自的 `noise` 是组内。**第 11 课教你怎么把它摊薄，这一课教你怎么直接把它知道掉。**


## 五、更一般的招：控制变量

第四节要的是"你真的知道下雨几天"。可多数时候你手上不是一个确定数，而是**另一件你了解得更多的事**。这一节给它一个通用形式，也是原书第 4 章第一节的正式内容。

想法是这样。你要估一个量 $Y$（下个月单量），你手上还有一个量 $X$（上个月单量、或者平台后台已经给你的近三十天曝光），它满足两个条件：**和 $Y$ 相关**，而且**它自己的平均你知道**。那就不要单独估 $Y$，改估这个差：

$$Y - a \times (X - \text{已知的 } E[X])$$

它回答的问题是：**"我能不能少抽一点随机数，把抽过的部分重复利用？"** 三项读法：$Y$ 是你要的东西；$(X - E[X])$ 是"这件事这次比它平时的平均水平高了多少"；减掉 $a$ 倍，就是把这部分"你已经能从 $X$ 那边解释掉的运气"扣掉。**关键是它不减掉真值**：$X$ 的平均你已知、$(X - E[X])$ 长期为零，所以期望不变，只是散布变小。

$a$ 取多大有最优解（课文进阶框给三行推导），实测就用最优值。我把 $Y$ 和 $X$ 的起伏都归一为 1，只改相关度 $\rho$，各跑两万遍（`control` 已在最前面的定义格里造好）：


In [5]:
print("（辅助量和目标的起伏都归一为 1；a 取最优值 ρ）")
for rho in [0.0, 0.3, 0.5, 0.7, 0.9]:
    random.seed(7)
    before, after = control(rho, 20000)
    theory = math.sqrt(1 - rho ** 2)
    print(f"相关 {rho:.1f}：毛边 减前 {before:.3f} → 减后 {after:.3f}；理论 √(1−ρ²) = {theory:.3f}")


（辅助量和目标的起伏都归一为 1；a 取最优值 ρ）
相关 0.0：毛边 减前 0.997 → 减后 0.997；理论 √(1−ρ²) = 1.000
相关 0.3：毛边 减前 0.996 → 减后 0.951；理论 √(1−ρ²) = 0.954
相关 0.5：毛边 减前 0.996 → 减后 0.863；理论 √(1−ρ²) = 0.866
相关 0.7：毛边 减前 0.996 → 减后 0.712；理论 √(1−ρ²) = 0.714
相关 0.9：毛边 减前 0.997 → 减后 0.435；理论 √(1−ρ²) = 0.436


理论那一列是 $\sqrt{1 - \rho^2}$。这张表有一个非常容易被忽略的性质：**起作用的是相关的平方。**

- $\rho = 0.3$（"多少有点关系"），毛边只降 **4.5%**；
- $\rho = 0.5$，降 13.4%；
- $\rho = 0.7$，降 28.5%；
- $\rho = 0.9$，降 **56.4%**。


把第四节那一块和第五节这张表画在同一张图上，这一节课的两条线就合成一条了：左边那根柱子告诉你**该去找信息还是该加次数**，右边那条下弯的曲线告诉你**什么样的辅助量才值得动手**。

![左格：一天的起伏平方拆成两段，蓝色组内 30.4%（36.7）、橙色组间 69.6%（84.0），整月晃 60.2 单，把天气定下来只剩 33.2 单。右格：横轴是辅助量和目标的相关度 ρ，纵轴是减完之后还剩多少毛边；蓝线是理论 √(1−ρ²)，橙点是两万遍实测，ρ=0.3 只降 4.5%，ρ=0.9 才降 56.4%](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig12_conditioning_and_control.svg)

图由 [code/fig12_conditioning_and_control.py](../code/fig12_conditioning_and_control.py) 生成，橙点用的就是上面那张表那一轮抽签（种子 7、各两万遍）。

所以这一招的真实脾气是：**找个"有点相关"的辅助量基本白干，找个几乎同涨同跌的辅助量才能把误差腰斩。** 原书第 4 章的技术含量，很大一部分就花在"到哪儿去找那个高相关的辅助量"。它给的第一答案，对你来说其实很眼熟：**就用你自己那个粗糙的旧估计。** 第 09 课 bootstrap 出来的均值、第 10 课那个中位数、你去年拍脑袋定的那版预算——它们和你要算的新数字高度相关（同一个世界、同一批记录），而它们的平均你自己就能算。这一条到第 14 课还会再出现一次：那时"旧估计"换成一个能用现成公式直接算出来的近似价格，相关度能高到表格最右端。


## 六、两种用法会把这招用废

**第一种：组平均是你自己估的，那部分毛边演多少遍都不会小。**

第四节整段都假定"雨天平均 60 单"是已知的。可现实里它是从你历史里那几条雨天记录算的。假设只有 3 条雨天记录，跑两万遍看这个"雨天平均"自己飘多少：


In [6]:
for m in [3, 8, 30]:
    random.seed(7)
    means = []
    for _ in range(20000):
        sample = [random.gauss(RAIN_MEAN, RAIN_SD) for _ in range(m)]
        means.append(sum(sample) / m)
    print(f"用 {m:>2} 条雨天记录估'雨天平均单量'：毛边 {spread(means):.2f} 单；乘到 {rain_days} 天上 → ±{spread(means) * rain_days:.1f} 单")


用  3 条雨天记录估'雨天平均单量'：毛边 4.62 单；乘到 12 天上 → ±55.4 单
用  8 条雨天记录估'雨天平均单量'：毛边 2.81 单；乘到 12 天上 → ±33.8 单
用 30 条雨天记录估'雨天平均单量'：毛边 1.46 单；乘到 12 天上 → ±17.5 单


看第一行。你把天气定下来，本可以省下第三节的 25 单毛边；**可你用 3 条记录去定"雨天平均"，光是这件事就带进 ±55 单的不确定——比你省下的多一倍。** 分组分得太细，比不分组更糟。

更要命的是这一列毛边的性质：它是**估计量的毛边**，不随模拟次数减少。你把两万遍改成两百万遍，4.62 一分不会降。这正是第 07 课那个区分：**多演治手抖，治不了你量不准的那把尺。**

所以定组之前先数一数每组有几条记录：只有两三条的组，不值得为它单开一个条件。


**第二种："已知"必须是你当时真的知道。**

第三节我用的是"预报给你 12 天雨"。如果那个 12 其实是你用 $30 \times 0.3$ 算出来的，你并不是知道它，你只是把随机性强行抹平了——你报出的 ±35 会变成一种不存在的安全感，而真实毛边还是 60。第 09 课第六节第三条讲过分位数的伪精度，这里是它的姊妹：**用期望冒充已知，得到的是偏窄的区间**（均值不偏，你的信心偏）。

同一道理，**未来的信息不能拿来条件**。跑完两万遍再回头看"哪几遍的下雨天数接近后来真实发生的那个月"，挑那些遍报数——那不是条件期望，那是事后挑数据，属于第 07 课那个"瞄准镜歪"。条件期望合法的唯一形态是：**在开始抽之前，你就已经知道。**


## 七、今晚就能用起来

代码在 [code/lesson12_conditioning_and_control.py](../code/lesson12_conditioning_and_control.py)。把它改成你自己的场景，只需要动最上面那五个数（`RAIN_P`、两组的平均和起伏、`DAYS`）。

三件可以直接做的事：

**第一件：把已知项从模拟里拿出去。** 你算下个月家庭现金流时，房租、房贷、地铁月票、订阅费这些**根本不该出现在随机抽签里**，它们是加法里的确定项。把第 04 课那段月度模拟拿来照这一条改一遍：固定项先加，只让真正会晃的那几项参与抽签。你会立刻发现自己的"最坏情况"没那么吓人，因为原来它被你把确定项一起抖进去了。

**第二件：先决定"下个月发几条"。** 这是第 09 课那个播放量模拟的直接升级。下面这一段做了这个对照：七条历史播放量不变，一条把发文条数钉成 22，一条让条数在 14 到 30 之间随机（`random.randint(14, 30)` 抽一个整数，和 `uniform` 一回事，只是不出小数）：


In [7]:
plays = [820, 1200, 3400, 610, 950, 2100, 480]
random.seed(7)
fixed22 = [sum(random.choices(plays, k=22)) for _ in range(20000)]
random.seed(7)
n_unknown = [sum(random.choices(plays, k=random.randint(14, 30))) for _ in range(20000)]
print(f"已知下个月发 22 条：月总播放 平均 {sum(fixed22) / len(fixed22):.0f}，起伏 {spread(fixed22):.0f}")
print(f"条数在 14 到 30 之间随机：平均 {sum(n_unknown) / len(n_unknown):.0f}，起伏 {spread(n_unknown):.0f}")
print(f"期望几乎没变，起伏多了 {(spread(n_unknown) / spread(fixed22) - 1) * 100:.1f}%")


已知下个月发 22 条：月总播放 平均 30027，起伏 4510
条数在 14 到 30 之间随机：平均 30080，起伏 8027
期望几乎没变，起伏多了 78.0%


**期望几乎一样，起伏差了 78%。** 也就是说，逼自己先定一个发文条数（哪怕定错两条），比多跑十万遍模拟有用得多。这一条大概是这一节课对你今天最直接的改变。

**第三件：给别人提要求时，问"这块误差是组内还是组间"。** 你的方案被质疑"不确定太大"时，先把它拆成两块：能从现实里查清楚的那块（组间）去找信息，剩下的才需要加模拟次数。这两件事的预算和周期完全不同，混在一起说，别人只会让你"再多跑点数据"。


## 这一课带走的一句话

知道的事就别再抽：把已知定下来，期望要重算、毛边往往能少四成；而想靠辅助量压误差，得找到几乎同涨同跌的那个，因为省下的比例是相关的平方。

配套课文：[第 12 课（网页版）](../docs/m0-foundations/12-conditioning-and-control.md)


---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。
